In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
### package imports
import numpy as np

# fileops
from pathlib import Path

# spatial
import xarray as xa
import geopandas as gpd

# plotting
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib as mpl
import cartopy.crs as ccrs
import cartopy.feature as cfeature

# notebook helpers
from python_helpers import stocks, map_panels, data_processing

ModuleNotFoundError: No module named 'numpy'

In [ ]:
### configuration

# filepaths
DATA_DIR_FP = Path("data")
OUTPUT_DIR_FP = Path("output")
SEAGRASS_EOV_FP = DATA_DIR_FP / "seagrass_eov"
# original data
all_poly_fp = SEAGRASS_EOV_FP / "seagrass_eov_poly_2025.csv"
all_points_fp = SEAGRASS_EOV_FP / "seagrass_eov_points_2025.csv"
# chosen model predictions
pred_fp = (
    OUTPUT_DIR_FP / "pixel_grouped_51-53-56-73-77/predictions/gpr_prediction_grid.nc"
)
PIXEL_GROUPED_DIR = pred_fp.parents[1]
STOCKS_DIR = PIXEL_GROUPED_DIR / "stocks"
CACHE_DIR = STOCKS_DIR / "cache"
STOCKS_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)

# plotting
EUROPE_LON_RANGE = (-8.85, 33.30)
EUROPE_LAT_RANGE = (34.65, 60.11)
LAND_COLOUR = "#e5e5e5"
BORDER_COLOUR = "#a6a6a6"
RIVER_COLOUR = "royalblue"
DPI = 300
SPECIES_COLOR_MAP = {
    "Cymodocea nodosa": "#1b9e77",
    "Halophila stipulacea": "#d95f02",
    "Posidonia oceanica": "#66a61e",
    "Zostera marina": "#a6761d",
    "Zostera marina and Cymodocea nodosa": "#e6ab02",
    "Zostera noltei": "#e7298a",
    "Zostera marina and Zostera noltei": "#666666",
    "Unspecified": "black",
}
# map seagrass_species to prediction_species_index from species-specific model predictions
SPECIES_INDEX_MAP = {
    1: "Cymodocea nodosa",
    2: "Halophila stipulacea",
    3: "Posidonia oceanica",
    4: "Unspecified",
    5: "Zostera marina",
    6: "Zostera marina and Cymodocea nodosa",
    7: "Zostera noltei",
}
INV_SPECIES_INDEX_MAP = {v: k for k, v in SPECIES_INDEX_MAP.items()}

In [ ]:
### Load species-specific prediction grid
pred_ds = xa.open_dataset(pred_fp)

In [ ]:
# Geometry attribution to seagrass species and seagrass EOV dataframe prep
polygon_attributed_fp = STOCKS_DIR / "seagrass_eov_gdf_attributed.parquet"
point_attributed_fp = STOCKS_DIR / "seagrass_eov_pts_attributed.parquet"
poly_df_fp = STOCKS_DIR / "seagrass_eov_poly_2025_complete.parquet"
points_df_fp = STOCKS_DIR / "seagrass_eov_points_2025_complete.parquet"

gdf_attributed, pts_attributed = data_processing.load_or_build_attributed_geometries(
    pred_ds=pred_ds,
    attributed_polygon_fp=polygon_attributed_fp,
    attributed_points_fp=point_attributed_fp,
    all_poly_fp=all_poly_fp,
    all_points_fp=all_points_fp,
    poly_cache_fp=poly_df_fp,
    points_cache_fp=points_df_fp,
    inv_species_index_map=INV_SPECIES_INDEX_MAP,
    lons=EUROPE_LON_RANGE,
    lats=EUROPE_LAT_RANGE,
)

display(
    gdf_attributed[
        ["seagrass_species", "prediction_species_index", "gC_cm3", "gC_cm3_se"]
    ].head(5)
)
display(
    pts_attributed[
        ["seagrass_species", "prediction_species_index", "gC_cm3", "gC_cm3_se"]
    ].head(5)
)

In [ ]:
# Mapping of polygons and points to EEZs

gdf_fp = STOCKS_DIR / "gdf_attributed_joined.parquet"
pts_fp = STOCKS_DIR / "points_attributed_joined.parquet"

gdf_attributed_joined, pts_attributed_joined = data_processing.build_or_load_eez_joins(
    gdf_attributed=gdf_attributed,
    pts_attributed=pts_attributed,
    eez_fp=DATA_DIR_FP / "eezs" / "World_EEZ_v12_20231025/eez_v12.shp",
    polygon_join_fp=gdf_fp,
    point_join_fp=pts_fp,
    buffer_deg=0.2,
    polygon_chunk_size=3000,
    point_chunk_size=10000,
)

In [ ]:
### visualise the EEZs
eez_gdf = gpd.read_file(DATA_DIR_FP / "eezs" / "World_EEZ_v12_20231025/eez_v12.shp")
# crop to Europe
europe_eez_gdf = eez_gdf.cx[
    EUROPE_LON_RANGE[0] : EUROPE_LON_RANGE[1], EUROPE_LAT_RANGE[0] : EUROPE_LAT_RANGE[1]
]

fig, ax = plt.subplots(
    figsize=(10, 10), dpi=DPI, subplot_kw={"projection": ccrs.PlateCarree()}
)

europe_eez_gdf.plot(ax=ax, color="#9EF8EE", edgecolor="black", alpha=0.5)
ax.add_feature(cfeature.LAND, color=LAND_COLOUR)
ax.add_feature(cfeature.BORDERS, color=BORDER_COLOUR, alpha=1, zorder=0, lw=0.5)

ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.set_extent(
    [EUROPE_LON_RANGE[0], EUROPE_LON_RANGE[1], EUROPE_LAT_RANGE[0], EUROPE_LAT_RANGE[1]]
)

In [ ]:
# caluclate carbon stocks, excluding Tunisia (large, undetailed polygons which skew results) and Libya (no environmental covariates)
ea_gdf_attributed_joined, pts_attributed_joined = (
    data_processing.prepare_european_stock_tables(
        gdf_attributed_joined,
        pts_attributed_joined,
        excluded_sovereign=["Tunisia", "Libya"],
        equal_area_epsg=3035,
    )
)
# add area and stock columns for national summaries
ea_gdf_attributed_joined = data_processing.add_stock_columns(
    ea_gdf_attributed_joined, depth_cm=100.0
)

# combine with the Gomis study data
combined_study_df = data_processing.build_comparison_dataframe(
    ea_gdf_attributed_joined,
    SEAGRASS_EOV_FP / "Gomis_2025_ST3.csv",
)

stocks.plot_comparison_bars(
    combined_study_df,
    density_error_metric="stock_kg_ha_se_upper",
    stock_error_metric="stock_Gg_se_upper",
    dpi=DPI,
)

In [ ]:
g = ea_gdf_attributed_joined.copy()
# All territories × species; territories sorted by total stock (see stocks.plot_territory_species_stacked_all).
fig, ax, stacked_stocks_GC = stocks.plot_territory_species_stacked_all(
    g,
    species_color_map=SPECIES_COLOR_MAP,
    dpi=DPI,
    ylim=(0, 7e4),
    show_species_errorbars=True,
    error_method="rss",
)
plt.show()


In [ ]:
gomis_multiplier = np.nansum(combined_study_df["stock_Gg"]) / np.nansum(
    combined_study_df["Biomass carbon stock (Gg)"]
)
print(f"Number of times greater than Gomis study stock: {gomis_multiplier:.2f}")

In [ ]:
# Stacked species contribution by territory (top N vs remainder), with total-stock error bars.
g_TgC = g.copy()

g_TgC["stock_Tg"] = g_TgC["stock_Gg"] / 1e3
g_TgC["stock_Tg_se"] = g_TgC["stock_Gg_se"] / 1e3

fig, (ax_l, ax_r) = stocks.plot_territory_species_stacked_split(
    g,
    ylabel="Carbon stock (GgC)",
    se_col="stock_Gg_se",
    error_method="upper",  # or "rss"
    species_color_map=SPECIES_COLOR_MAP,
    dpi=DPI,
    top_n=5,
    ylim_left=(0, 6e4),
    ylim_right=(0, 910),
)
plt.show()

In [ ]:
fig, (ax_l, ax_r) = stocks.plot_territory_species_stacked_split(
    g_TgC,
    species_color_map=SPECIES_COLOR_MAP,
    se_col="stock_Gg_se",
    error_method="upper",
    show_species_errorbars=True,
    dpi=DPI,
    top_n=5,
    ylim_left=(0, 6e4),
    ylim_right=(0, 910),
)
plt.show()


In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
g = ea_gdf_attributed_joined.copy()

fig, ax, grouped_stocks_GC = stocks.plot_territory_species_grouped_all(
    g,
    species_color_map=SPECIES_COLOR_MAP,
    dpi=DPI,
    ylim=(0, 6e4),
    error_method="upper",        # matches split-panel figures; use "rss" for narrower bounds
    errorbar_color="black",
    region_outline_color="black",
    region_errorbar_linewidth=2.5,
    species_errorbar_linewidth=2.5,
    species_errorbar_capsize=4.5,
)
plt.show()

In [ ]:
g = ea_gdf_attributed_joined.copy()

fig, (ax_l, ax_r) = stocks.plot_territory_species_grouped_split(
    g,
    species_color_map=SPECIES_COLOR_MAP,
    dpi=DPI,
    error_method="upper",
    top_n=5,
    ylim_left=(0, 6e4),
    ylim_right=(0, 910),
    ylabel="Carbon stock (GgC)",
    errorbar_color="black",
    region_outline_color="black",
    region_errorbar_linewidth=2.5,
    species_errorbar_linewidth=2.5,
    species_errorbar_capsize=4.5,
)
plt.show()

In [ ]:
fig, (ax_l, ax_r) = stocks.plot_territory_species_stacked_split(
    g_TgC,
    species_color_map=SPECIES_COLOR_MAP,
    se_col="stock_Gg_se",
    error_method="upper",
    show_species_errorbars=True,
    dpi=DPI,
    top_n=5,
    ylim_left=(0, 6e4),
    ylim_right=(0, 800),
)
plt.show()

In [ ]:
# calculate the total stock across all territories from combined_study_df
total_stock = combined_study_df["stock_Gg"].sum()
total_stock_se = (combined_study_df["stock_Gg_se_upper"] ** 2).sum() ** 0.5
print(f"Total stock: {total_stock:.2f} GgC, Error: {total_stock_se:.2f} GgC")

In [ ]:
fig, ax = plt.subplots(figsize=(12, 3), dpi=DPI)

values = stacked_stocks_GC.sum(axis=0)

ax.set_axisbelow(True)
ax.grid(True, which="major", axis="x", alpha=0.5)
ax.grid(True, which="minor", axis="x", alpha=0.1)

# order values by magnitude and colour by SPECIES_COLOR_MAP
values = values.sort_values(ascending=True)
ax.barh(
    values.index,
    values.values,
    color=[SPECIES_COLOR_MAP.get(s, "#777777") for s in values.index],
    height=0.5,
)
plt.xscale("log")
plt.xlabel("Carbon Stock (GgC)")
# plt.ylabel("Seagrass Species")
plt.xticks(rotation=45)
plt.show()

In [ ]:
# plot a stacked bar chart by species type, ordered by total area
# Pivot table: index = territory, columns = species, values = area
stacked_areas = (
    g.groupby(["TERRITORY1", "seagrass_species"])["area_ha"].sum().unstack(fill_value=0)
)
# Order by total area
stacked_areas = stacked_areas.loc[
    stacked_areas.sum(axis=1).sort_values(ascending=False).index
]

# Reorder columns according to SPECIES_COLOR_MAP if possible
species_order = [c for c in SPECIES_COLOR_MAP if c in stacked_areas.columns] + [
    c for c in stacked_areas.columns if c not in SPECIES_COLOR_MAP
]
stacked_areas = stacked_areas[species_order]

fig, ax = plt.subplots(figsize=(12, 5), dpi=DPI)
stacked_areas.plot(
    kind="bar",
    stacked=True,
    ax=ax,
    color=[SPECIES_COLOR_MAP.get(s, "#777777") for s in stacked_areas.columns],
    edgecolor="none",
)
plt.xticks(rotation=45)
plt.yscale("log")
ax.set_axisbelow(True)
ax.grid(True, which="major", axis="y", alpha=0.5)
ax.grid(True, which="minor", axis="y", alpha=0.1)
ax.set_xlabel("National Territory")
ax.set_ylabel("Area (ha)")
# replace " " with "\n" in the x-axis labels
xlabels = ax.get_xticklabels()
xlabels = [label.get_text().replace(" ", "\n") for label in xlabels]
ax.set_xticklabels(xlabels, ha="center")
handles, labels = ax.get_legend_handles_labels()
legend = ax.legend(
    handles,
    labels,
    title="Seagrass Species",
    bbox_to_anchor=(0.5, 1.1),
    ncols=len(stacked_areas.columns),
    loc="center",
    borderaxespad=0.0,
)
# Set the legend title font to bold
if legend.get_title() is not None:
    legend.get_title().set_fontweight("bold")

plt.tight_layout()
plt.show()

# Polygon mapping

In [ ]:
### configure European seagrass polygon mapping
EXTENTS = {
    # "Europe": (-8.85, 33.3, 34.64, 60.11),
    "Europe": (-12, 33.3, 34.64, 60.11),
    # "Crete": (23.5, 27.1, 34.5, 35.95),
    # "UK": (-6, 1.5, 49, 51.5),
    # "Sicily": (10.5, 17, 36, 39.5),
    # "Baltic": (5, 13.5, 54, 59.5),
    # "Tyrrhenian": (7, 11, 39, 43),
    # "Spain": (-4, 3, 36, 42),
    # "Aegean": (21, 28, 35.5, 41),
}  # lon_min, lon_max, lat_min, lat_max
TOLERANCES = {
    "Europe": 0.00001,
    "Crete": 0.0000001,
    "Baltic": 0.000001,
    "Scotland": 0.000001,
    "Tyrrhenian": 0.000001,
    "Aegean": 0.000001,
    "UK": 0.000001,
    "Spain": 0.000001,
    "Sicily": 0.000001,
}  # Manual overrides to ensure visibility of polygons
BUFFERS = {
    "Europe": 0.08,
    "Crete": 0.03,
    "Baltic": 0.08,
    "Tyrrhenian": 0.03,
    "Aegean": 0.05,
    "UK": 0.03,
    "Spain": 0.06,
    "Sicily": 0.03,
}  # Manual overrides to ensure visibility of polygons
USE_LOG_COLORBAR = True

In [ ]:
# ============================================================
# Figure: multi-panel map with polygon + point geometries only
# Color shows attributed carbon density on a logarithmic colorbar
# ============================================================


poly_gdf = map_panels.to_wgs84_clean(ea_gdf_attributed_joined)
pt_gdf = map_panels.to_wgs84_clean(pts_attributed_joined)
poly_val_col = "stock_Gg"
pt_val_col = poly_val_col  # N.B. points can't have stocks, only density

# if poly_val_col not in poly_gdf.columns or pt_val_col not in pt_gdf.columns:
if poly_val_col not in poly_gdf.columns:
    raise KeyError(f"Missing plotting column: {poly_val_col}")

print("Preparing panel data...")
simplified = {
    name: map_panels.get_simplified(
        name,
        poly_gdf,
        extent,
        cache_dir=CACHE_DIR,
        tolerance=TOLERANCES.get(name),
        buffer=BUFFERS.get(name, 0.0),
    ).loc[lambda d: d[poly_val_col].notna()]
    for name, extent in EXTENTS.items()
}
print("All data ready.\n")

if poly_val_col == "stock_Gg":
    vmin_log, vmax = 1.0, 1e4
    cbar_label = "Estimated median carbon stock (GgC) in the top metre of sediment"
    log_ticks = [1, 10, 100, 1000]
elif poly_val_col == "gC_cm3":
    vmin_log, vmax = 1e-4, 0.25
    cbar_label = "Estimated median carbon density (gC/cm³) in the top metre of sediment"
    log_ticks = [1e-4, 1e-3, 1e-2, 1e-1, 0.25]
elif poly_val_col == "stock_kg_ha":
    vmin_log, vmax = 1.0, 1e4
    cbar_label = "Estimated median carbon stock (GgC/ha) in the top metre of sediment"
    log_ticks = [0.01, 0.1, 1, 10, 100, 1000]
else:
    raise ValueError(f"Invalid polygon value column: {poly_val_col}")

norm = (
    mpl.colors.LogNorm(vmin=vmin_log, vmax=vmax)
    if USE_LOG_COLORBAR
    else mpl.colors.Normalize(vmin=0.0, vmax=vmax)
)
cmap = "plasma_r"

# ------------------------------------------------------------------
# Figure layout (same structure as original multi-panel map)
# ------------------------------------------------------------------
fig = plt.figure(figsize=(13.5, 7.8), dpi=250)
outer = gridspec.GridSpec(
    2,
    2,
    figure=fig,
    width_ratios=[1.5, 1],
    height_ratios=[1.5, 1],
    left=0.025,
    right=0.995,
    bottom=0.08,  # leave room for colorbar
    top=0.99,
    wspace=0.015,
    hspace=0.015,
)

ax_main = fig.add_subplot(outer[0, 0], projection=ccrs.PlateCarree())
right_stack = outer[0, 1].subgridspec(3, 1, hspace=0.015)
ax_r1 = fig.add_subplot(right_stack[0, 0], projection=ccrs.PlateCarree())
ax_r2 = fig.add_subplot(right_stack[1, 0], projection=ccrs.PlateCarree())
ax_r3 = fig.add_subplot(right_stack[2, 0], projection=ccrs.PlateCarree())

bottom_stack = outer[1, 0].subgridspec(1, 3, hspace=0.015)
# To create non-equal width for the bottom row, redefine bottom_stack with custom width_ratios:
bottom_stack = outer[1, 0].subgridspec(1, 3, wspace=0.015, width_ratios=[1, 1, 1.5])
ax_b1 = fig.add_subplot(bottom_stack[0, 0], projection=ccrs.PlateCarree())
ax_b2 = fig.add_subplot(bottom_stack[0, 1], projection=ccrs.PlateCarree())
ax_b3 = fig.add_subplot(bottom_stack[0, 2], projection=ccrs.PlateCarree())
ax_br = fig.add_subplot(outer[1, 1], projection=ccrs.PlateCarree())

panel_specs = [
    (
        ax_main,
        "Europe",
        "Europe",
        dict(show_left=True, show_top=True, show_labels=True),
    ),
    # (ax_r1, "Crete", "Crete", {}),
    # (ax_r2, "UK", "South-west UK", {}),
    # (ax_r3, "Sicily", "Sicily", {}),
    # (ax_b1, "Baltic", "North/Baltic Sea", {}),
    # (ax_b2, "Tyrrhenian", "Tyrrhenian Sea", {}),
    # (ax_b3, "Spain", "Eastern Spain", {}),
    # (ax_br, "Aegean", "Aegean Sea", {}),
]

for ax, name, title, kwargs in panel_specs:
    if name not in EXTENTS:
        continue
    map_panels.draw_geom_panel(
        ax,
        name,
        EXTENTS[name],
        title,
        simplified=simplified,
        pt_gdf=pt_gdf,
        poly_val_col=poly_val_col,
        pt_val_col=pt_val_col,
        cmap=cmap,
        norm=norm,
        land_colour=LAND_COLOUR,
        border_colour=BORDER_COLOUR,
        river_colour=RIVER_COLOUR,
        **kwargs,
    )

sm = mpl.cm.ScalarMappable(norm=norm, cmap=cmap)
cbar_ax = fig.add_axes([0.05, -0.01, 0.9, 0.025])
cbar = fig.colorbar(sm, cax=cbar_ax, orientation="horizontal", extend="both", pad=0.08)
cbar.set_label(cbar_label, fontsize=14)
cbar.ax.tick_params(labelsize=10)

if USE_LOG_COLORBAR:
    cbar.set_ticks(log_ticks)
    cbar.set_ticklabels(log_ticks, fontsize=10)
elif poly_val_col == "gC_cm3":
    lin_ticks = [0, 0.05, 0.10, 0.15, 0.20, 0.25]
    cbar.set_ticks(lin_ticks)
    cbar.ax.set_xticklabels([f"{v:.2f}" for v in lin_ticks], fontsize=10)

In [ ]:
### configure European seagrass polygon mapping
EXTENTS = {
    "Europe": (-8.85, 33.3, 34.64, 60.11),
    "Crete": (23.5, 27.1, 34.5, 35.95),
    "UK": (-6, 1.5, 49, 51.5),
    "Sicily": (10.5, 17, 36, 39.5),
    "Baltic": (5, 13.5, 54, 59.5),
    "Tyrrhenian": (7, 11, 39, 43),
    "Spain": (-4, 3, 36, 42),
    "Aegean": (21, 28, 35.5, 41),
}  # lon_min, lon_max, lat_min, lat_max
TOLERANCES = {
    "Europe": 0.00001,
    "Crete": 0.0000001,
    "Baltic": 0.000001,
    "Scotland": 0.000001,
    "Tyrrhenian": 0.000001,
    "Aegean": 0.000001,
    "UK": 0.000001,
    "Spain": 0.000001,
    "Sicily": 0.000001,
}  # Manual overrides to ensure visibility of polygons
BUFFERS = {
    "Europe": 0.08,
    "Crete": 0.03,
    "Baltic": 0.08,
    "Tyrrhenian": 0.03,
    "Aegean": 0.05,
    "UK": 0.03,
    "Spain": 0.06,
    "Sicily": 0.03,
}  # Manual overrides to ensure visibility of polygons
USE_LOG_COLORBAR = False

In [ ]:
# ============================================================
# Figure: multi-panel map with polygon + point geometries only
# Color shows attributed carbon density on a logarithmic colorbar
# ============================================================


poly_gdf = map_panels.to_wgs84_clean(ea_gdf_attributed_joined)
poly_gdf["stock_Gg_percent_error"] = (
    100 * poly_gdf["stock_Gg_se"] / poly_gdf["stock_Gg"]
)
pt_gdf = map_panels.to_wgs84_clean(pts_attributed_joined)
# poly_val_col = "stock_Gg_se"
poly_val_col = "stock_Gg_percent_error"

pt_val_col = poly_val_col  # N.B. points can't have stocks, only density

# if poly_val_col not in poly_gdf.columns or pt_val_col not in pt_gdf.columns:
if poly_val_col not in poly_gdf.columns:
    raise KeyError(f"Missing plotting column: {poly_val_col}")

print("Preparing panel data...")
simplified = {
    name: map_panels.get_simplified(
        name,
        poly_gdf,
        extent,
        cache_dir=CACHE_DIR,
        tolerance=TOLERANCES.get(name),
        buffer=BUFFERS.get(name, 0.0),
    ).loc[lambda d: d[poly_val_col].notna()]
    for name, extent in EXTENTS.items()
}
print("All data ready.\n")

if poly_val_col == "stock_Gg" or poly_val_col == "stock_Gg_se":
    vmin_log, vmax = 1.0, 1e4
    cbar_label = "Estimated median carbon stock (GgC) in the top metre of sediment"
    log_ticks = [1, 10, 100, 1000]
elif poly_val_col == "gC_cm3" or poly_val_col == "gC_cm3_se":
    vmin_log, vmax = 1e-4, 0.25
    cbar_label = "Estimated median carbon density (gC/cm³) in the top metre of sediment"
    log_ticks = [1e-4, 1e-3, 1e-2, 1e-1, 0.25]
elif poly_val_col == "stock_kg_ha" or poly_val_col == "stock_kg_ha_se":
    vmin_log, vmax = 1.0, 1e4
    cbar_label = "Estimated median carbon stock (GgC/ha) in the top metre of sediment"
    log_ticks = [0.01, 0.1, 1, 10, 100, 1000]
elif poly_val_col == "stock_Gg_percent_error":
    vmin_log, vmax = 10, 100
    log_ticks = [1e-3, 1e-2, 1e-1, 1, 10, 100]
    cbar_label = "Percentage error in estimated median carbon stock (GgC) in the top metre of sediment"
else:
    raise ValueError(f"Invalid polygon value column: {poly_val_col}")

norm = (
    mpl.colors.LogNorm(vmin=vmin_log, vmax=vmax)
    if USE_LOG_COLORBAR
    else mpl.colors.Normalize(vmin=0.0, vmax=vmax)
)
cmap = "plasma_r"

# ------------------------------------------------------------------
# Figure layout (same structure as original multi-panel map)
# ------------------------------------------------------------------
fig = plt.figure(figsize=(13.5, 7.8), dpi=250)
outer = gridspec.GridSpec(
    2,
    2,
    figure=fig,
    width_ratios=[1.5, 1],
    height_ratios=[1.5, 1],
    left=0.025,
    right=0.995,
    bottom=0.08,  # leave room for colorbar
    top=0.99,
    wspace=0.015,
    hspace=0.015,
)

ax_main = fig.add_subplot(outer[0, 0], projection=ccrs.PlateCarree())
right_stack = outer[0, 1].subgridspec(3, 1, hspace=0.015)
ax_r1 = fig.add_subplot(right_stack[0, 0], projection=ccrs.PlateCarree())
ax_r2 = fig.add_subplot(right_stack[1, 0], projection=ccrs.PlateCarree())
ax_r3 = fig.add_subplot(right_stack[2, 0], projection=ccrs.PlateCarree())

bottom_stack = outer[1, 0].subgridspec(1, 3, hspace=0.015)
# To create non-equal width for the bottom row, redefine bottom_stack with custom width_ratios:
bottom_stack = outer[1, 0].subgridspec(1, 3, wspace=0.015, width_ratios=[1, 1, 1.5])
ax_b1 = fig.add_subplot(bottom_stack[0, 0], projection=ccrs.PlateCarree())
ax_b2 = fig.add_subplot(bottom_stack[0, 1], projection=ccrs.PlateCarree())
ax_b3 = fig.add_subplot(bottom_stack[0, 2], projection=ccrs.PlateCarree())
ax_br = fig.add_subplot(outer[1, 1], projection=ccrs.PlateCarree())

panel_specs = [
    (
        ax_main,
        "Europe",
        "Europe",
        dict(show_left=True, show_top=True, show_labels=True),
    ),
    (ax_r1, "Crete", "Crete", {}),
    (ax_r2, "UK", "South-west UK", {}),
    (ax_r3, "Sicily", "Sicily", {}),
    (ax_b1, "Baltic", "North/Baltic Sea", {}),
    (ax_b2, "Tyrrhenian", "Tyrrhenian Sea", {}),
    (ax_b3, "Spain", "Eastern Spain", {}),
    (ax_br, "Aegean", "Aegean Sea", {}),
]

for ax, name, title, kwargs in panel_specs:
    if name not in EXTENTS:
        continue
    map_panels.draw_geom_panel(
        ax,
        name,
        EXTENTS[name],
        title,
        simplified=simplified,
        pt_gdf=pt_gdf,
        # rivers_gdf=rivers_gdf,
        poly_val_col=poly_val_col,
        pt_val_col=pt_val_col,
        cmap=cmap,
        norm=norm,
        land_colour=LAND_COLOUR,
        border_colour=BORDER_COLOUR,
        river_colour=RIVER_COLOUR,
        **kwargs,
    )

sm = mpl.cm.ScalarMappable(norm=norm, cmap=cmap)
cbar_ax = fig.add_axes([0.05, -0.01, 0.9, 0.025])
cbar = fig.colorbar(sm, cax=cbar_ax, orientation="horizontal", pad=0.08)
cbar.set_label(cbar_label, fontsize=14)
cbar.ax.tick_params(labelsize=10)

if USE_LOG_COLORBAR:
    cbar.set_ticks(log_ticks)
    cbar.set_ticklabels(log_ticks, fontsize=10)
elif poly_val_col == "gC_cm3":
    lin_ticks = [0, 0.05, 0.10, 0.15, 0.20, 0.25]
    cbar.set_ticks(lin_ticks)
    cbar.ax.set_xticklabels([f"{v:.2f}" for v in lin_ticks], fontsize=10)


In [ ]:
# plot polygons coloured by SOVEREIGN1
# convert to epsg 4326
g_map = g.to_crs(epsg=4326)

national_buffered_gdf = map_panels.get_simplified(
    "Europe_national",
    g_map,
    [*EUROPE_LON_RANGE, *EUROPE_LAT_RANGE],
    cache_dir=CACHE_DIR,
    tolerance=TOLERANCES.get("Europe"),
    buffer=BUFFERS.get("Europe", 0.0),
)

fig, ax = plt.subplots(
    figsize=(12, 5), dpi=DPI, subplot_kw={"projection": ccrs.PlateCarree()}
)
ax.coastlines(linewidth=0.3, color="0.35")
ax.add_feature(cfeature.LAND, color=LAND_COLOUR, alpha=1, zorder=0)
ax.add_feature(cfeature.BORDERS, color=BORDER_COLOUR, alpha=1, zorder=0, lw=0.5)

national_buffered_gdf.iloc[::1].plot(
    ax=ax,
    column="TERRITORY1",
    cmap="tab20b",
    legend=True,
    legend_kwds={"loc": "center left", "bbox_to_anchor": (1.03, 0.5)},
)

map_panels.style_map_axes(
    ax,
    [*EUROPE_LON_RANGE, *EUROPE_LAT_RANGE],
    show_bottom=False,
    show_left=True,
    show_top=True,
    show_right=False,
    show_labels=True,
    target_ticks=4,
)

europe_eez_gdf.plot(ax=ax, color="#9EF8EE", edgecolor=BORDER_COLOUR, alpha=0.1)

for spine in ax.spines.values():
    spine.set_linewidth(1.0)
    spine.set_edgecolor("0.15")
